In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, shutil, hashlib, subprocess, glob
from pathlib import Path
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
subprocess.run(['git','pull','--ff-only','--quiet'],check=False)
import importlib
if 'config' in sys.modules: importlib.reload(sys.modules['config'])
import config
import numpy as np, pandas as pd
print('ready:', os.getcwd())


Mounted at /content/drive
ready: /content/drive/MyDrive/CALSHIFT_Research/calshift-research


In [2]:
# =============================================================================
# Cell 2 - NSL reconstruction, matching nb03/nb05 exactly.
# Cached probs: probs_{arch}_s{seed}.npz with S_pool / target / probcal in
# canonical order Normal,DoS,Probe,R2L,U2R. Ladder D_eval subsets come from
# nslkdd_ladder_assignments.parquet. Focal = R2L. We use the HIGHEST rung (0.80)
# as NSL's comparison point, for parity with CIC (hardest holdout) and UGR
# (single shift) - recorded choice. Deterministic mid-U APS score (matches
# nb20/nb21); nb05 randomized APS remains the coverage of record.
# =============================================================================
ALPHA=config.ALPHA_PRIMARY
CLASSES=config.CANONICAL_CLASSES; c2i={c:i for i,c in enumerate(CLASSES)}; K=len(CLASSES)
FOCAL='R2L'; RUNG_USED=0.80

nsl_train=pd.read_parquet(config.INTERIM_DIR/'nslkdd_train.parquet').reset_index(drop=True)
nsl_test =pd.read_parquet(config.INTERIM_DIR/'nslkdd_test.parquet').reset_index(drop=True)
part=pd.read_parquet(config.PROC_DIR/'nslkdd_source_partition_labels.parquet')
nsl_train=nsl_train.assign(partition=part['partition'].values)
y_sp=nsl_train[nsl_train.partition=='source_cal_pool']['label'].map(c2i).to_numpy()
y_te=nsl_test['label'].map(c2i).to_numpy()

assign=pd.read_parquet(config.PROC_DIR/'nslkdd_ladder_assignments.parquet')
IDX={(r,j,role):g['test_idx'].to_numpy() for (r,j,role),g in assign.groupby(['rung','realization','role'])}
REALS=sorted(assign[assign.rung==RUNG_USED]['realization'].unique())
MODELS=sorted(glob.glob(str(config.PROC_DIR/'probs_*.npz')))
print(f'S_pool={len(y_sp)} target={len(y_te)} | rung {RUNG_USED} realizations={len(REALS)} | models={len(MODELS)}')

def dmidU_scores(P):  # deterministic mid-U APS score per class (matches nb20/nb21)
    order=np.argsort(-P,axis=1); sp=np.take_along_axis(P,order,1); cum=np.cumsum(sp,1)
    ss=cum-0.5*sp; out=np.empty_like(P); np.put_along_axis(out,order,ss,1); return out
def qhat(s,a):
    n=len(s); return np.inf if n<1 else float(np.quantile(s,min(np.ceil((n+1)*(1-a))/n,1.0),method='higher'))
def ks(a,b):
    if len(a)<5 or len(b)<5: return np.nan
    allv=np.sort(np.concatenate([a,b]))
    ca=np.searchsorted(np.sort(a),allv,side='right')/len(a); cb=np.searchsorted(np.sort(b),allv,side='right')/len(b)
    return float(np.max(np.abs(ca-cb)))
print('helpers ready; focal', FOCAL)


S_pool=18894 target=22544 | rung 0.8 realizations=20 | models=30
helpers ready; focal R2L


In [3]:
# =============================================================================
# Cell 3 - seed-average NSL calibrated probs PER ARCHITECTURE (no cross-arch
# blend), on S_pool and the full target pool. Then pool the rung-0.80 D_eval
# subsets across realizations into one target evaluation set per architecture.
# =============================================================================
ARCHS=['rf','xgb','mlp']
def load_arch(arch):
    fs=sorted(glob.glob(str(config.PROC_DIR/f'probs_{arch}_s*.npz')))
    sp=tg=None; sh=None; k=0
    for f in fs:
        z=np.load(f); s,t=z['S_pool'],z['target']
        if sh is None: sh=(s.shape,t.shape)
        elif (s.shape,t.shape)!=sh: raise ValueError(f'shape mismatch {Path(f).name}')
        sp=s if sp is None else sp+s; tg=t if tg is None else tg+t; k+=1
    return (sp/k).astype(np.float64),(tg/k).astype(np.float64),k

# rung-0.80 D_eval indices, pooled across the 20 realizations (unique rows)
eval_idx=np.unique(np.concatenate([IDX[(RUNG_USED,j,'eval')] for j in REALS]))
print('pooled rung-0.80 D_eval rows (unique):', len(eval_idx))
ARCH_P={}
for arch in ARCHS:
    Psp,Ptg,k=load_arch(arch); ARCH_P[arch]=(Psp,Ptg)
    print(f'  {arch}: averaged {k} seeds | S_pool{Psp.shape} target{Ptg.shape}')


pooled rung-0.80 D_eval rows (unique): 16301
  rf: averaged 10 seeds | S_pool(18894, 5) target(22544, 5)
  xgb: averaged 10 seeds | S_pool(18894, 5) target(22544, 5)
  mlp: averaged 10 seeds | S_pool(18894, 5) target(22544, 5)


In [4]:
# =============================================================================
# Cell 4 - VALIDATION: recompute NSL focal coverage from these probs (mid-U,
# rung 0.80, SHC) and sanity-check the ordering against the committed
# coverage_primary_nslkdd.csv (which is randomized-APS, so numbers are close,
# not identical). We require SHC focal to be well below nominal (the known
# R2L collapse), and REC focal near nominal. Fail loud otherwise.
# =============================================================================
committed=pd.read_csv(config.REPORTS_DIR/'coverage_primary_nslkdd.csv')
comm=committed[(committed['class']==FOCAL)&(committed.alpha==ALPHA)&(committed.rung==RUNG_USED)&
               (committed.score=='aps')&(committed.variant=='mondrian')&(committed.feasible)]
comm_shc=float(comm[comm.protocol=='SHC']['coverage'].mean())
comm_rec=float(comm[comm.protocol=='REC']['coverage'].mean())
print(f'committed (randomized APS) rung {RUNG_USED} focal {FOCAL}: SHC={comm_shc:.3f}  REC={comm_rec:.3f}')

def shc_focal_cov(arch):
    Psp,Ptg=ARCH_P[arch]; Ssp=dmidU_scores(Psp); Ste=dmidU_scores(Ptg)
    fi=c2i[FOCAL]
    q=qhat(Ssp[y_sp==fi,fi],ALPHA)                 # SHC quantile from source true-focal
    ev=eval_idx; yev=y_te[ev]; sev=Ste[ev]
    m=yev==fi
    return float(np.mean(sev[m,fi]<=q)) if m.any() else np.nan
mine={a:shc_focal_cov(a) for a in ARCHS}
print('recomputed (mid-U) SHC focal coverage by arch:', {a:round(v,3) for a,v in mine.items()})
mean_mine=float(np.nanmean(list(mine.values())))
print(f'mean recomputed SHC focal = {mean_mine:.3f}  (committed randomized = {comm_shc:.3f})')
assert mean_mine < (1-ALPHA)-0.10, 'VALIDATION FAIL: expected R2L SHC collapse well below nominal'
if not np.isnan(comm_shc) and abs(mean_mine-comm_shc) > 0.15:
    print(f'  WARNING: recomputed ({mean_mine:.3f}) diverges from committed ({comm_shc:.3f}) by '
          f'{abs(mean_mine-comm_shc):.3f} (>0.15). mid-U vs randomized differ, but this much gap '
          f'is worth inspecting before trusting NSL rows.')
else:
    print('VALIDATION PASS: R2L collapses under SHC, consistent with the committed results.')


committed (randomized APS) rung 0.8 focal R2L: SHC=0.030  REC=0.956
recomputed (mid-U) SHC focal coverage by arch: {'rf': 0.056, 'xgb': 0.047, 'mlp': 0.117}
mean recomputed SHC focal = 0.074  (committed randomized = 0.030)
VALIDATION PASS: R2L collapses under SHC, consistent with the committed results.


In [5]:
# =============================================================================
# Cell 5 - build NSL rows for BOTH tables (mechanism nb20 + monitor nb21) and
# APPEND to the committed CSVs. Per class: score movement (mechanism), label-free
# predicted-class drift + misroute (monitor), true undercoverage. Per architecture.
# =============================================================================
MIN_SUPPORT=20
mech_rows=[]; mon_rows=[]
ev=eval_idx; yev=y_te[ev]
for arch in ARCHS:
    Psp,Ptg=ARCH_P[arch]; Ssp=dmidU_scores(Psp); Ste=dmidU_scores(Ptg)
    yhat_sp=np.argmax(Psp,1); yhat_te=np.argmax(Ptg[ev],1)
    for ci,cn in enumerate(CLASSES):
        # ---- mechanism (true-class score movement, source vs target) ----
        s_src_true=Ssp[y_sp==ci,ci]; s_tgt_true=Ste[ev][yev==ci,ci]
        if len(s_src_true)<5 or len(s_tgt_true)<5: continue
        q=qhat(s_src_true,ALPHA); shc=float(np.mean(s_tgt_true<=q)); under=(1-ALPHA)-shc
        med=float(np.median(s_tgt_true)-np.median(s_src_true))
        q90=float(np.quantile(s_tgt_true,0.9)-np.quantile(s_src_true,0.9))
        ks_true=ks(s_src_true,s_tgt_true)
        mech_rows.append({'dataset':f'nslkdd:rung{RUNG_USED}','arch':arch,'class':cn,
            'n_src':len(s_src_true),'n_tgt':len(s_tgt_true),'q_src':round(q,4),
            'SHC_coverage':round(shc,4),'undercoverage':round(under,4),
            'median_score_shift':round(med,4),'q90_score_shift':round(q90,4),'score_KS':round(ks_true,4)})
        # ---- monitor (label-free predicted-class drift + misroute) ----
        s_src_pred=Ssp[yhat_sp==ci,ci]; s_tgt_pred=Ste[ev][yhat_te==ci,ci]
        n_pred_tgt=int((yhat_te==ci).sum()); n_pred_src=int((yhat_sp==ci).sum())
        low=(n_pred_tgt<MIN_SUPPORT) or (n_pred_src<MIN_SUPPORT)
        drift_lf=np.nan if low else ks(s_src_pred,s_tgt_pred)
        share_s=n_pred_src/max(len(Psp),1); share_t=n_pred_tgt/max(len(ev),1)
        pmd=float(share_s-share_t)
        misroute=float(np.mean(yhat_te[yev==ci]!=ci)) if (yev==ci).any() else np.nan
        mon_rows.append({'dataset':f'nslkdd:rung{RUNG_USED}','arch':arch,'class':cn,
            'n_pred_target':n_pred_tgt,'low_support':bool(low),
            'drift_labelfree':(np.nan if (isinstance(drift_lf,float) and np.isnan(drift_lf)) else round(drift_lf,4)),
            'pred_mass_drop':round(pmd,4),'drift_truelabel':round(ks_true,4),
            'misroute_target':round(misroute,4),'SHC_coverage':round(shc,4),'undercoverage':round(under,4)})

nsl_mech=pd.DataFrame(mech_rows); nsl_mon=pd.DataFrame(mon_rows)
print('NSL mechanism rows:', len(nsl_mech), '| NSL monitor rows:', len(nsl_mon))
print('\nNSL monitor (mean over archs):')
print(nsl_mon.groupby('class').agg(drift_labelfree=('drift_labelfree','mean'),
      drift_truelabel=('drift_truelabel','mean'),misroute=('misroute_target','mean'),
      undercoverage=('undercoverage','mean')).round(4).to_string())

# append to the committed tables (idempotent: drop any prior nslkdd rows first)
mech_csv=config.REPORTS_DIR/'score_shift_explainability.csv'
mon_csv =config.REPORTS_DIR/'monitor_labelfree.csv'
me=pd.read_csv(mech_csv); mo=pd.read_csv(mon_csv)
me=me[~me['dataset'].astype(str).str.startswith('nslkdd')]
mo=mo[~mo['dataset'].astype(str).str.startswith('nslkdd')]
me=pd.concat([me,nsl_mech],ignore_index=True); mo=pd.concat([mo,nsl_mon],ignore_index=True)
me.to_csv(mech_csv,index=False); mo.to_csv(mon_csv,index=False)
print(f'\nappended -> {mech_csv.name} now {len(me)} rows ; {mon_csv.name} now {len(mo)} rows')


NSL mechanism rows: 15 | NSL monitor rows: 15

NSL monitor (mean over archs):
        drift_labelfree  drift_truelabel  misroute  undercoverage
class                                                            
DoS              0.2480           0.3647    0.3653         0.3225
Normal           0.1992           0.0646    0.0273        -0.0227
Probe            0.3840           0.3774    0.3696         0.3196
R2L              0.8357           0.9216    0.9648         0.8765
U2R                 NaN           0.5973    0.8406         0.4138

appended -> score_shift_explainability.csv now 60 rows ; monitor_labelfree.csv now 60 rows


In [6]:
# =============================================================================
# Cell 6 - re-run the headline tests on ALL THREE datasets now that NSL is in,
# and commit. This confirms the mechanism and the monitor hold with NSL included.
# =============================================================================
from scipy import stats
from sklearn.metrics import roc_auc_score
me=pd.read_csv(config.REPORTS_DIR/'score_shift_explainability.csv')
mo=pd.read_csv(config.REPORTS_DIR/'monitor_labelfree.csv')

rho_m,p_m=stats.spearmanr(me['score_KS'],me['undercoverage'])
print(f'MECHANISM all 3 datasets: Spearman(score_KS, undercoverage) rho={rho_m:+.3f} p={p_m:.3g} (n={len(me)})')

mo2=mo.copy(); mo2['is_under']=(mo2['undercoverage']>0.05).astype(int)
r_d=mo2['drift_labelfree'].rank(pct=True); r_m=mo2['pred_mass_drop'].clip(lower=0).rank(pct=True)
mo2['lf_detector']=np.where(mo2['low_support'].fillna(False),r_m,r_d)
rho_lf,p_lf=stats.spearmanr(mo2['lf_detector'],mo2['undercoverage'])
auc=roc_auc_score(mo2['is_under'],mo2['lf_detector']) if mo2['is_under'].nunique()>1 else np.nan
print(f'MONITOR all 3 datasets: detector rho={rho_lf:+.3f} p={p_lf:.3g} | AUC={auc:.3f} | undercovering {int(mo2.is_under.sum())}/{len(mo2)}')
print('\nby dataset (monitor detector rho):')
for ds in ['nslkdd','cicids2017','ugr16']:
    sub=mo2[mo2.dataset.str.startswith(ds)]
    if len(sub)>3:
        rr,_=stats.spearmanr(sub['lf_detector'],sub['undercoverage']); print(f'  {ds:11s} rho={rr:+.3f} (n={len(sub)})')

allverdict={'note':'NSL-KDD added to mechanism (nb20) and monitor (nb21) at rung 0.80',
    'mechanism_spearman_KS_all3':{'rho':round(float(rho_m),4),'p':float(p_m),'n':int(len(me))},
    'monitor_detector_all3':{'rho':round(float(rho_lf),4),'p':float(p_lf),
        'auc':(None if np.isnan(auc) else round(float(auc),4)),'n':int(len(mo2))},
    'nsl_rung_used':RUNG_USED,'nsl_focal':FOCAL}
(config.REPORTS_DIR/'nsl_added_verdict.json').write_text(json.dumps(allverdict,indent=2))
print('\n',json.dumps(allverdict,indent=2))

def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT); git('add','-A',show=False)
if git('status','--porcelain',show=False).stdout.strip():
    git('commit','-m','nb22: add NSL-KDD (rung 0.80) to mechanism + monitor tables; headline tests hold on all 3 datasets')
    r=git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)


MECHANISM all 3 datasets: Spearman(score_KS, undercoverage) rho=+0.925 p=4.45e-26 (n=60)
MONITOR all 3 datasets: detector rho=+0.725 p=5.64e-11 | AUC=0.895 | undercovering 32/60

by dataset (monitor detector rho):
  nslkdd      rho=+0.384 (n=15)
  cicids2017  rho=+0.768 (n=30)
  ugr16       rho=+0.824 (n=15)

 {
  "note": "NSL-KDD added to mechanism (nb20) and monitor (nb21) at rung 0.80",
  "mechanism_spearman_KS_all3": {
    "rho": 0.9252,
    "p": 4.454468796260778e-26,
    "n": 60
  },
  "monitor_detector_all3": {
    "rho": 0.7252,
    "p": 5.641349490136993e-11,
    "auc": 0.8951,
    "n": 60
  },
  "nsl_rung_used": 0.8,
  "nsl_focal": "R2L"
}
[main a8c1d16] nb22: add NSL-KDD (rung 0.80) to mechanism + monitor tables; headline tests hold on all 3 datasets
 5 files changed, 48 insertions(+), 1 deletion(-)
 create mode 100644 notebooks/22_add_nsl_to_mechanism_monitor.ipynb
 create mode 100644 reports/nsl_added_verdict.json
Branch 'main' set up to track remote branch 'main' from 'or